In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Fix the random seed so every run gives the same result
np.random.seed(123)

# Generate a small synthetic dataset
n = 80  # number of samples (change this to experiment)
x = np.linspace(-3, 3, n).reshape(-1, 1)  # input feature x: n points from -3 to 3
t = 0.5 * x.ravel()**2 + x.ravel()  # true (noise-free) quadratic curve
y = t + np.random.randn(n) * 0.6  # observed data = true curve + noise

# Fit a polynomial of degree `deg` and plot the result
def plot_polynomial_fit(x, t, y, deg):
    """
    Fit a polynomial to the data and plot three curves:
    the original data, the fitted polynomial, and the ideal curve.

    Parameters
    ----------
    x : ndarray
        x-coordinates of the data
    y : ndarray
        y-coordinates of the observed (noisy) data
    deg : int
        Degree of the polynomial
    """
    # np.polyfit: least-squares fit -> polynomial coefficients
    # np.poly1d:  turn the coefficients into a polynomial function
    x_flat = x.ravel()  # flatten x to 1-D (required by np.polyfit)
    p = np.poly1d(np.polyfit(x_flat, y, deg))  # fit the polynomial


    # Red dots: observed data | blue line: fitted polynomial | red dashed: ideal curve
    plt.plot(x, y, 'ro', label='Original Data')
    plt.plot(x, p(x), '-', label=f'Degree {deg} Fit')
    plt.plot(x, 0.5 * x_flat**2 + x_flat, 'r--', label='Ideal Result')

    plt.legend()  # show the legend

# Degree 1: too simple -> underfitting
# Degree 3: good fit
# Degree 10: follows the noise too closely -> overfitting
plt.figure(figsize=(18, 4), dpi=200)
degrees = [1, 3, 10]  # polynomial degrees to try
titles = ['Under Fitting', 'Fitting', 'Over Fitting']  # titles of the three subplots
for index, deg in enumerate(degrees):
    plt.subplot(1, 3, index + 1)  # 1 row, 3 columns, position index+1
    plot_polynomial_fit(x, t, y, deg)
    plt.title(titles[index], fontsize=20)

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error


# Goal: reduce overfitting with regularization.
# Degree 10 overfits (shown above), so we compare:
#   - no regularization
#   - L1 (Lasso), L2 (Ridge), ElasticNet (L1 + L2 combined)
degree = 10  # high degree -> overfitting baseline
# PolynomialFeatures: creates x, x^2, ..., x^10 from the input
# include_bias=False: avoid a duplicate intercept with the linear model
models = {
    'No regularization': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        LinearRegression()
    ),
    'L1 (Lasso)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Lasso(alpha=0.01, max_iter=100000)  # alpha: strength of the L1 penalty
    ),
    'L2 (Ridge)': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        Ridge(alpha=0.1)  # alpha: strength of the L2 penalty
    ),
    'ElasticNet': make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100000)  # combines L1 and L2
    )
}

plt. figure(figsize=(10, 6))  # create a new figure
# Train each model and plot its predicted curve
for name, model in models.items():
    model.fit(x, y)  # train on the observed data

    y_pred = model.predict(x)  # predict on the same x values

    # train_mse: how well the model fits the noisy data y
    # test_mse:  how close the prediction is to the ideal curve t
    train_mse = mean_squared_error(y, y_pred)
    test_mse = mean_squared_error(t, y_pred)

    plt.plot(x, y_pred, '--', linewidth=2,
             label=f'{name} (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
# Ideal (noise-free) curve for reference
plt.plot(x, 0.5 * x.ravel()**2 + x.ravel(), 'k-', linewidth=2, label='Ideal Result')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Comparison of Different Regularization Methods under Overfitting (Degree=10)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ==================== Dropout & Early Stopping ====================
# Dropout and early stopping are regularization methods for neural networks.
# They do not fit into the sklearn Pipeline above, so we use Keras
# to build a neural network that overfits easily, then show how each method helps.
import matplotlib.pyplot as plt  # plotting library
import tensorflow as tf  # deep learning framework (preinstalled in Colab)
from tensorflow.keras.models import Sequential  # builds a network layer by layer
from tensorflow.keras.layers import Dense, Dropout  # Dense: fully connected layer; Dropout: randomly turns off neurons
from tensorflow.keras.callbacks import EarlyStopping  # stops training when validation loss stops improving
from sklearn.preprocessing import StandardScaler  # rescales the input data
from sklearn.metrics import mean_squared_error  # metric used to evaluate predictions

# Neural networks train better on standardized inputs (mean 0, std 1)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)  # fit: compute mean/std, then transform x
y_2d = y.reshape(-1, 1)  # Keras expects y as a 2-D array: (n_samples, 1)


def build_mlp(with_dropout, dropout_rate=0.3, seed=0):
    """Build a large MLP; if with_dropout=True, add Dropout layers between hidden layers."""
    tf.keras.utils.set_random_seed(seed)  # same seed for every model -> fair comparison
    model = Sequential()
    model.add(Dense(128, activation='relu', input_shape=(1,)))  # hidden layer 1: 128 neurons
    if with_dropout:
        model.add(Dropout(dropout_rate))  # randomly deactivate 30% of neurons during training
    model.add(Dense(128, activation='relu'))  # hidden layer 2: 128 neurons
    if with_dropout:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))  # output layer: 1 neuron -> predicted y (regression)
    model.compile(optimizer='adam', loss='mse')  # optimizer: adam; loss: mean squared error
    return model


# 2x2 comparison: Dropout (yes/no) x early stopping (yes/no)
#   1) No Dropout + early stopping
#   2) Dropout(0.3) + early stopping
#   3) Dropout(0.3) + no early stopping
#   4) No Dropout + no early stopping (full 200 epochs) -> baseline
epochs_dict = {}  # actual number of epochs each model trained for
results = {}  # name -> (trained model, training history, used early stopping or not)
for name, use_dropout, use_early_stop in [
        ('No Dropout, Early Stopping', False, True),
        ('Dropout (0.3), Early Stopping', True, True),
        ('No Dropout, No Early Stopping', False, False),
        ('Dropout (0.3), No Early Stopping', True, False)]:
    model = build_mlp(with_dropout=use_dropout)
    callbacks = []  # callbacks run during training
    if use_early_stop:
        # Early stopping: if validation loss does not improve for 50 epochs,
        # stop training and restore the best weights found so far
        callbacks.append(EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True))
    # Training settings:
    #   epochs=200          max training rounds
    #   validation_split=0.2 keep 20% of data as the validation set
    #   batch_size=16       update weights every 16 samples
    #   verbose=0           no progress output
    history = model.fit(x_scaled, y_2d, epochs=200, validation_split=0.2,
                        batch_size=16, callbacks=callbacks, verbose=0)
    epochs_dict[name] = len(history.history["loss"])  # record how many epochs were actually trained
    results[name] = (model, history, use_early_stop)


# Compare the four predicted curves (same train/test MSE as the sklearn part)
plt.figure(figsize=(10, 6))
for name, (model, _, _) in results.items():
    y_pred = model.predict(scaler.transform(x), verbose=0).ravel()  # standardize x, predict, flatten to 1-D
    train_mse = mean_squared_error(y, y_pred)  # error vs noisy data
    test_mse = mean_squared_error(t, y_pred)  # error vs ideal curve -> how well it generalizes
    plt.plot(x, y_pred, '--', linewidth=2,
             label=f'{name}: trained {epochs_dict.get(name)} epchos (train_MSE: {train_mse:.3f}, test_MSE: {test_mse:.3f})')
plt.plot(x, t, 'k-', linewidth=2, label='Ideal Result')  # reference: ideal curve
plt.xlabel('x')
plt.ylabel('y')
plt.title('Dropout & Early Stopping Regularization (MLP)')
plt.legend(loc='best', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
